# LCNC Platform Adoption — Statistical Analysis
## Factors Affecting Behavioral Intention to Use Low-Code/No-Code Platforms
### Based on UTAUT2 Framework

---

| Item | Detail |
|---|---|
| **Dataset** | `LowCode_NoCode.sav` (SPSS format) |
| **N** | ~568 respondents (411 after listwise deletion) |
| **Scale** | 5-point Likert |
| **Dependent Variable** | BI_T — Behavioral Intention |
| **Independent Variables** | PE, EE, SI, HM, HB, FC, PC, TA, PT |

---

## 0. Install Required Libraries

Run this cell once if packages are not yet installed.

In [ ]:
# Install required packages (run once)
import sys
!{sys.executable} -m pip install pyreadstat pandas numpy scipy statsmodels scikit-learn matplotlib seaborn pingouin -q

## 1. Import Libraries & Set Paths

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pyreadstat
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
import json
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120  # Screen display quality
plt.rcParams['savefig.dpi'] = 400  # Save quality

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH   = "/Users/wirapongc/Library/CloudStorage/GoogleDrive-wirach@kku.ac.th/My Drive/OpenClawMacMini/LCNC/2026/LowCode_NoCode.sav"
OUTPUT_DIR  = Path("/Users/wirapongc/Library/CloudStorage/GoogleDrive-wirach@kku.ac.th/My Drive/OpenClawMacMini/LCNC/2026/Analysis_Output/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print("Libraries loaded ✅")

---
## Step 1: Data Loading & Preparation

- Load `.sav` file using `pyreadstat`
- Check for missing values
- Compute composite (mean) scores for each construct
- Apply listwise deletion

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────
df, meta = pyreadstat.read_sav(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

# ── Check missing values ───────────────────────────────────────────────────
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("\n✅ No missing values found.")
else:
    print(f"\n⚠️  Missing values detected:")
    print(missing_cols)

# ── Compute composite scores ───────────────────────────────────────────────
constructs = {
    'BI_T': ['BI1', 'BI2', 'BI3'],
    'PE_T': ['PE1', 'PE2', 'PE3'],
    'EE_T': ['EE1', 'EE2', 'EE3'],
    'SI_T': ['SI1', 'SI2', 'SI3'],
    'HM_T': ['HM1', 'HM2', 'HM3'],
    'HB_T': ['HB1', 'HB2', 'HB3'],
    'FC_T': ['FC1', 'FC2', 'FC3'],
    'PC_T': ['PC1', 'PC2', 'PC3'],
    'TA_T': ['TA1', 'TA2', 'TA3'],
    'PT_T': ['PT1', 'PT2', 'PT3'],
}

print("\nComputing composite scores:")
for composite, items in constructs.items():
    available = [c for c in items if c in df.columns]
    if available:
        df[composite] = df[available].mean(axis=1)
        print(f"  ✅ {composite} = mean({available})")
    else:
        print(f"  ❌ WARNING: Items not found for {composite}: {items}")

# ── Listwise deletion ──────────────────────────────────────────────────────
composite_vars = list(constructs.keys())
df_clean = df.dropna(subset=composite_vars).copy()
print(f"\nOriginal N = {df.shape[0]}")
print(f"After listwise deletion N = {df_clean.shape[0]}  (dropped {df.shape[0] - df_clean.shape[0]})")

---
## Step 2: Descriptive Statistics

- Mean, SD, Min, Max for all composite variables
- Frequency distributions for demographic variables

In [ ]:
# ── Composite variable descriptives ───────────────────────────────────────
print("Descriptive Statistics — Composite Variables")
print("-" * 55)

desc_rows = []
for var in composite_vars:
    desc_rows.append({
        'Variable': var,
        'N': int(df_clean[var].count()),
        'Mean': df_clean[var].mean(),
        'SD': df_clean[var].std(),
        'Min': df_clean[var].min(),
        'Max': df_clean[var].max(),
    })

desc_df = pd.DataFrame(desc_rows).set_index('Variable')
desc_df = desc_df.round(3)
display(desc_df)

In [ ]:
# ── Demographic frequency distributions ───────────────────────────────────
demo_vars = ['GENDER', 'AGE', 'EXPERIENCE', 'AFFILIATION']

for dv in demo_vars:
    if dv in df_clean.columns:
        counts = df_clean[dv].value_counts().sort_index()
        total = counts.sum()
        rows = []
        for val, cnt in counts.items():
            label = str(val)
            if hasattr(meta, 'variable_value_labels') and dv in meta.variable_value_labels:
                label = meta.variable_value_labels[dv].get(val, str(val))
            rows.append({'Label': label, 'N': cnt, '%': round(cnt / total * 100, 1)})
        rows.append({'Label': 'Total', 'N': total, '%': 100.0})
        print(f"\nFrequency Distribution: {dv}")
        display(pd.DataFrame(rows).set_index('Label'))

---
## Step 3: Reliability Analysis — Cronbach's Alpha

- Acceptable threshold: α ≥ 0.70
- Good: α ≥ 0.80

In [ ]:
construct_names = {
    'BI': 'Behavioral Intention',
    'PE': 'Performance Expectancy',
    'EE': 'Effort Expectancy',
    'SI': 'Social Influence',
    'HM': 'Hedonic Motivation',
    'HB': 'Habit',
    'FC': 'Facilitating Conditions',
    'PC': 'Price Consciousness',
    'TA': 'Techno-Anxiety',
    'PT': 'Privacy/Trust',
}

alpha_data = {}
alpha_rows = []
all_items = []

for abbr, full_name in construct_names.items():
    items = [f"{abbr}{i}" for i in range(1, 4)]
    available = [c for c in items if c in df_clean.columns]
    if len(available) >= 2:
        sub = df_clean[available].dropna()
        try:
            alpha_val = pg.cronbach_alpha(data=sub)[0]
        except:
            k = len(available)
            iv = sub.var(ddof=1)
            tv = sub.sum(axis=1).var(ddof=1)
            alpha_val = (k / (k - 1)) * (1 - iv.sum() / tv)
        alpha_data[abbr] = alpha_val
        status = "✅ Good" if alpha_val >= 0.80 else ("✅ Acceptable" if alpha_val >= 0.70 else "⚠️ Low")
        alpha_rows.append({'Construct': full_name, 'Code': abbr, 'Items': str(available), 'Alpha': round(alpha_val, 3), 'Status': status})
        all_items.extend(available)

# Overall alpha
all_items_avail = [c for c in all_items if c in df_clean.columns]
sub_all = df_clean[all_items_avail].dropna()
try:
    overall_alpha = pg.cronbach_alpha(data=sub_all)[0]
except:
    k = len(all_items_avail)
    iv = sub_all.var(ddof=1)
    tv = sub_all.sum(axis=1).var(ddof=1)
    overall_alpha = (k / (k - 1)) * (1 - iv.sum() / tv)
alpha_rows.append({'Construct': '── OVERALL SCALE ──', 'Code': 'ALL', 'Items': f'All {len(all_items_avail)} items', 'Alpha': round(overall_alpha, 3), 'Status': '✅ Excellent' if overall_alpha >= 0.90 else '✅ Good'})

alpha_df = pd.DataFrame(alpha_rows).set_index('Construct')
display(alpha_df)

---
## Step 4: Pearson Correlation Analysis

- Pearson correlation matrix for all IVs and DV
- Significance flags: * p<.05, ** p<.01, *** p<.001
- Multicollinearity check: flag r > 0.80 between IVs

In [ ]:
# ── Correlation matrix ─────────────────────────────────────────────────────
corr_vars   = composite_vars
corr_matrix = df_clean[corr_vars].corr(method='pearson')

print("Pearson Correlation Matrix (all constructs):")
display(corr_matrix.round(3))

# ── Correlations with BI_T ─────────────────────────────────────────────────
print("\nCorrelations with BI_T (Behavioral Intention):")
bi_corr_rows = []
for var in corr_vars:
    if var != 'BI_T':
        r, p = stats.pearsonr(df_clean['BI_T'], df_clean[var])
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        bi_corr_rows.append({'Predictor': var, 'r': round(r, 3), 'p-value': round(p, 4), 'Sig': sig})
bi_corr_df = pd.DataFrame(bi_corr_rows).sort_values('r', ascending=False).set_index('Predictor')
display(bi_corr_df)

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))

mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.3f',
    cmap='RdYlGn', center=0, square=True,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8},
    annot_kws={'size': 9}
)
ax.set_title(
    'Pearson Correlation Matrix\n(Low-Code/No-Code Platform Adoption Factors)',
    fontsize=14, fontweight='bold', pad=20
)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

heatmap_path = OUTPUT_DIR / 'correlation_heatmap.png'
plt.savefig(heatmap_path, dpi=400, bbox_inches='tight')
plt.show()
print(f"Saved: {heatmap_path}")

---
## Step 5: Multiple Linear Regression (OLS)

**Model:** `BI_T ~ PE_T + EE_T + SI_T + HM_T + HB_T + FC_T + PC_T + TA_T + PT_T`

Reports:
- Unstandardized (B) and Standardized (Beta) coefficients
- SE, t-statistic, p-value
- R, R², Adjusted R², F-statistic
- VIF (multicollinearity check)

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
predictors = ['PE_T', 'EE_T', 'SI_T', 'HM_T', 'HB_T', 'FC_T', 'PC_T', 'TA_T', 'PT_T']
outcome    = 'BI_T'

reg_data  = df_clean[[outcome] + predictors].dropna()
X         = reg_data[predictors]
y         = reg_data[outcome]
X_const   = sm.add_constant(X)

# ── OLS regression ─────────────────────────────────────────────────────────
model = sm.OLS(y, X_const).fit()

# ── Model summary ──────────────────────────────────────────────────────────
print("Model Summary")
print("-" * 40)
print(f"  R            = {np.sqrt(model.rsquared):.4f}")
print(f"  R²           = {model.rsquared:.4f}")
print(f"  Adjusted R²  = {model.rsquared_adj:.4f}")
print(f"  F-statistic  = {model.fvalue:.3f}")
print(f"  F p-value    = {model.f_pvalue:.6f}")
print(f"  AIC          = {model.aic:.3f}")
print(f"  BIC          = {model.bic:.3f}")
print(f"  N            = {len(reg_data)}")

In [ ]:
# ── Standardized coefficients ──────────────────────────────────────────────
X_std       = (X - X.mean()) / X.std()
y_std       = (y - y.mean()) / y.std()
model_std   = sm.OLS(y_std, sm.add_constant(X_std)).fit()

# ── Coefficient table ──────────────────────────────────────────────────────
coef_data = {}
coef_rows = []

for var in ['const'] + predictors:
    b  = model.params[var]
    se = model.bse[var]
    t  = model.tvalues[var]
    p  = model.pvalues[var]
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    if var == 'const':
        coef_rows.append({'Variable': '(Constant)', 'B': round(b, 3), 'SE': round(se, 3), 'Beta': '–', 't': round(t, 3), 'p': round(p, 4), 'Sig': sig})
    else:
        beta = model_std.params[var]
        coef_data[var] = {'B': b, 'SE': se, 'Beta': beta, 't': t, 'p': p, 'sig': sig}
        coef_rows.append({'Variable': var, 'B': round(b, 3), 'SE': round(se, 3), 'Beta': round(beta, 3), 't': round(t, 3), 'p': round(p, 4), 'Sig': sig})

coef_df = pd.DataFrame(coef_rows).set_index('Variable')
print("Regression Coefficients:")
display(coef_df)

In [ ]:
# ── VIF ────────────────────────────────────────────────────────────────────
vif_data = {}
vif_rows = []
for i, var in enumerate(predictors):
    vif = variance_inflation_factor(X_const.values, i + 1)
    vif_data[var] = vif
    status = '✅ OK' if vif < 5 else ('⚠️ Moderate' if vif < 10 else '❌ High')
    vif_rows.append({'Variable': var, 'VIF': round(vif, 3), 'Status': status})

vif_df = pd.DataFrame(vif_rows).set_index('Variable')
print("Variance Inflation Factor (VIF):")
display(vif_df)

# ── Regression equation ────────────────────────────────────────────────────
eq = f"BI_T = {model.params['const']:.3f}"
for var in predictors:
    b = model.params[var]
    sign = '+' if b >= 0 else '–'
    eq += f" {sign} {abs(b):.3f}×{var}"
print(f"\nRegression Equation:\n  {eq}")

---
## Step 6: Regression Assumption Diagnostics

Six diagnostic plots:
1. Residuals vs Fitted — linearity check
2. Normal Q-Q Plot — normality of residuals
3. Scale-Location — homoscedasticity
4. Residuals vs Leverage — influential outliers
5. Histogram of Residuals — bell-shaped distribution
6. Cook's Distance — influence of individual observations

In [ ]:
fitted    = model.fittedvalues
residuals = model.resid
std_resid = (residuals - residuals.mean()) / residuals.std()
influence = model.get_influence()
leverage  = influence.hat_matrix_diag
cooks_d   = influence.cooks_distance[0]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Regression Diagnostic Plots', fontsize=16, fontweight='bold')

# 1. Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, residuals, alpha=0.5, color='steelblue', s=20)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
z = np.polyfit(fitted, residuals, 2)
x_line = np.linspace(fitted.min(), fitted.max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), color='red', linewidth=1, alpha=0.7)
ax.set_xlabel('Fitted Values'); ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted'); ax.grid(True, alpha=0.3)

# 2. Q-Q Plot
ax = axes[0, 1]
(osm, osr), (slope, intercept, _) = stats.probplot(residuals, dist='norm')
ax.scatter(osm, osr, alpha=0.5, color='steelblue', s=20)
ax.plot(osm, slope * np.array(osm) + intercept, color='red', linewidth=1.5)
ax.set_xlabel('Theoretical Quantiles'); ax.set_ylabel('Sample Quantiles')
ax.set_title('Normal Q-Q Plot'); ax.grid(True, alpha=0.3)

# 3. Scale-Location
ax = axes[0, 2]
sqrt_abs = np.sqrt(np.abs(std_resid))
ax.scatter(fitted, sqrt_abs, alpha=0.5, color='steelblue', s=20)
z2 = np.polyfit(fitted, sqrt_abs, 1)
ax.plot(x_line, np.poly1d(z2)(x_line), color='red', linewidth=1.5)
ax.set_xlabel('Fitted Values'); ax.set_ylabel('√|Standardized Residuals|')
ax.set_title('Scale-Location'); ax.grid(True, alpha=0.3)

# 4. Residuals vs Leverage
ax = axes[1, 0]
ax.scatter(leverage, std_resid, alpha=0.5, color='steelblue', s=20)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1)
ax.axhline(y=2, color='orange', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(y=-2, color='orange', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Leverage'); ax.set_ylabel('Standardized Residuals')
ax.set_title('Residuals vs Leverage'); ax.grid(True, alpha=0.3)

# 5. Histogram of Residuals
ax = axes[1, 1]
ax.hist(residuals, bins=20, color='steelblue', edgecolor='white', alpha=0.7)
x_range = np.linspace(residuals.min(), residuals.max(), 100)
ax2 = ax.twinx()
ax2.plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()), 'r-', linewidth=2)
ax2.set_ylabel('Density', color='red')
ax.set_xlabel('Residuals'); ax.set_ylabel('Frequency')
ax.set_title('Histogram of Residuals'); ax.grid(True, alpha=0.3)

# 6. Cook's Distance
ax = axes[1, 2]
ax.bar(range(len(cooks_d)), cooks_d, color='steelblue', alpha=0.6)
threshold = 4 / len(reg_data)
ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5, label=f'4/n = {threshold:.4f}')
ax.set_xlabel('Observation Index'); ax.set_ylabel("Cook's Distance")
ax.set_title("Cook's Distance"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
diag_path = OUTPUT_DIR / 'diagnostic_plots.png'
plt.savefig(diag_path, dpi=400, bbox_inches='tight')
plt.show()
print(f"Saved: {diag_path}")

---
## Step 7: Save All Results

In [ ]:
# ── Save analysis_summary.json ─────────────────────────────────────────────
summary = {
    'N': int(len(reg_data)),
    'R': float(np.sqrt(model.rsquared)),
    'R2': float(model.rsquared),
    'Adj_R2': float(model.rsquared_adj),
    'F': float(model.fvalue),
    'F_df1': int(len(predictors)),
    'F_df2': int(len(reg_data) - len(predictors) - 1),
    'F_p': float(model.f_pvalue),
    'coefficients': {var: {k: float(v) if k != 'sig' else v for k, v in d.items()} for var, d in coef_data.items()},
    'vif': {k: float(v) for k, v in vif_data.items()},
    'alpha': {k: float(v) for k, v in alpha_data.items() if isinstance(v, (int, float))},
    'descriptives': {var: {'N': int(df_clean[var].count()), 'Mean': float(df_clean[var].mean()),
                           'SD': float(df_clean[var].std()), 'Min': float(df_clean[var].min()),
                           'Max': float(df_clean[var].max())} for var in composite_vars},
    'correlations_with_BI': {v: float(corr_matrix.loc['BI_T', v]) for v in corr_vars if v != 'BI_T'},
}

summary_path = OUTPUT_DIR / 'analysis_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ Saved: {summary_path}")

# ── Final summary ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("ANALYSIS COMPLETE ✅")
print("=" * 60)
print(f"  N                = {len(reg_data)}")
print(f"  R²               = {model.rsquared:.4f}")
print(f"  Adjusted R²      = {model.rsquared_adj:.4f}")
print(f"  F({len(predictors)}, {len(reg_data)-len(predictors)-1}) = {model.fvalue:.3f}, p < .001")
print("\nSignificant predictors (p < .05):")
for var, d in sorted(coef_data.items(), key=lambda x: abs(x[1]['Beta']), reverse=True):
    if d['p'] < 0.05:
        print(f"  {var}: B={d['B']:.3f}, Beta={d['Beta']:.3f}, p={d['p']:.4f} {d['sig']}")
print("\nNon-significant predictors:")
for var, d in coef_data.items():
    if d['p'] >= 0.05:
        print(f"  {var}: B={d['B']:.3f}, Beta={d['Beta']:.3f}, p={d['p']:.4f}")